# Functional dependencies and arithmetic identities

A functional dependency states that equal input values have equal output
values within the rows evaluated. The profiler screens candidate mappings,
searches combinations of columns, then verifies candidates. The `value_mapping`
form records a lookup relation; `affine` records a fitted equation such as
`y = a·x + b`. Both are functional dependencies.

## Semantics

The comparison excludes rows with a null in any input or target; `n_used`
records the rows evaluated. Values are compared in onceover's normalized
representation, not as formatted display strings. The specification records
this row-selection and comparison policy.

## Discovery: screen, lattice, verify

**Screen.** All eligible input and target columns are screened using a
2,048-row stratified sample. Determinants need at least two distinct values,
30 non-null rows, and a distinct ratio below 0.98. Near-unique columns are
handled by key analysis. The screen estimates violations for each input–target
pair. It retains up to 16 candidate determinants per target (32 on `deep`).

**Lattice.** Determinant sets grow level-wise — singletons, then pairs
(then triples on `deep`, drawn from the 12 columns that produced the most
exact dependencies). Subset pruning skips a candidate if one of its subsets already determines
the target. Determinants whose groups cover at least 95% of evaluated rows
are also excluded: such a near-unique input would appear to determine
almost any target.

**Verify.** Each candidate is rechecked using normalized values on up to 200,000
stratified rows (500,000 on `deep`). Hashing screens candidates; verification
determines whether they appear in this sample. A candidate that fails
becomes an approximate dependency if its error is small enough; a
candidate that passes is exact on the verified rows, and the finding says
which rows those were.

## Approximate dependencies

The reported g1 value is the fraction of evaluated rows that violate the
mapping. Approximate dependencies require g1 ≤ 0.01. The search retains at
most one per target, choosing the one with fewer violations and then fewer
inputs. It suppresses an approximate dependency if a subset of its inputs
already establishes the same output exactly. The finding reports the
violation count and evaluated-row count.

In [ ]:
import random

import polars as pl

import onceover

rng = random.Random(9)
n = 3_000
region_id = [rng.randrange(3) for _ in range(n)]
manager = [["Ann", "Bob", "Cid"][r] for r in region_id]
for i in (100, 900, 1700):          # exactly three violations
    manager[i] = "Dana"
df = pl.DataFrame({
    "region_id": region_id,
    "manager": manager,
    "noise": [rng.random() for _ in range(n)],
})

report = onceover.profile(df)
for f in report.findings:
    if f["category"] == "dependencies" and f.get("type") == "functional" and "manager" in f["subject"]["columns"]:
        e = f["evidence"][0]
        print(f"{f['claim']}")
        print(f"   violations={e['metrics']['violations']} n_used={e['metrics']['n_used']}")

## Arithmetic identities

The identity pass fits `y = a·x + b` by least squares on up to 20,000
sampled rows. It considers pairs with |Pearson| ≥ 0.8 or ξ ≥ 0.9, up to 100
(default) or 200 (`deep`). It checks each fitted equation on all
pair-complete rows, using a tolerance of 1e-9 absolute plus 1e-9 relative. An identity is reported when at
least 99.9% of rows match, with the coefficients, the tolerance, the
matching-row count, and any violating rows in the metrics. The statement
is `dependencies/functional` with form `affine`; the finding's
interpretation names the practical consequence — a derived column, or a
check column.

The pass checks pairwise affine relationships. It does not search for
multivariate formulas or nonlinear equations
([](../appendix/directions.ipynb)).

In [ ]:
df = pl.DataFrame({
    "quantity": [rng.randrange(1, 11) for _ in range(n)],
})
df = df.with_columns((2 * pl.col("quantity")).alias("double_quantity"))
report = onceover.profile(df)
identity = next(f for f in report.findings if f["specification"].get("form") == "affine")
print(identity["claim"])
print("a, b:", identity["specification"]["definition"]["a"], identity["specification"]["definition"]["b"])